In [ ]:
from pathlib import Path

import hardware
import torch
from env import SKILLS, DuckEnv
from IPython.display import Video
from model import ActorCritic
from onpolicy import OnPolicy, fit, flat, normalize, sample_rollout
from rollout import evaluate, record_video

In [ ]:
hw = hardware.load("2026-11-07-hd1910")
env = DuckEnv(SKILLS["walk"], hw, num_envs=2048, device="cuda", seed=1)
env.reset()
model = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")   # 61 个观测进，14 个关节的动作出

In [ ]:
rollout = sample_rollout(env, model, 24)                 # 每只鸭子走 24 步
{key: tuple(value.shape) for key, value in rollout.items()}

In [ ]:
def reward_to_go(rewards, dones, gamma):
    returns = torch.zeros_like(rewards)
    running = torch.zeros_like(rewards[0])
    for t in reversed(range(len(rewards))):              # 从最后一步往前算
        running = rewards[t] + gamma * running * (1 - dones[t])   # 回合在这一步结束就乘 0
        returns[t] = running
    return returns

In [ ]:
returns = reward_to_go(rollout["rewards"], rollout["dones"], 0.99)
returns[:, 0]                                            # 第一只鸭子这 24 步的回报

In [ ]:
class Reinforce(OnPolicy):
    gamma = 0.99
    entropy_coef = 0.01

    def prepare(self, rollout):
        returns = reward_to_go(rollout["rewards"], rollout["dones"], self.gamma)
        return {**flat(rollout, "obs", "actions"), "advantages": normalize(returns).flatten()}

    def loss(self, batch):
        dist = self.model.action_distribution(batch["obs"])
        entropy = dist.entropy().sum(-1).mean()
        return self.policy_loss(dist, batch) - self.entropy_coef * entropy   # 减一点熵，鼓励探索

    def policy_loss(self, dist, batch):
        log_probs = dist.log_prob(batch["actions"]).sum(-1)      # 14 个关节的对数概率相加
        return -(log_probs * batch["advantages"]).mean()

In [ ]:
before = evaluate(model.act_inference, SKILLS["walk"], hw, 0)
before["falls"], before["mean_step_reward"]

In [ ]:
fit(Reinforce(env, model), 1000)

In [ ]:
counter = env.unwrapped.common_step_counter               # 课程表走到的那一档，评测对齐到它
after = evaluate(model.act_inference, SKILLS["walk"], hw, counter)
after["falls"], after["mean_step_reward"]

In [ ]:
record_video(model.act_inference, SKILLS["walk"], hw, counter, Path("reinforce.mp4"))
Video("reinforce.mp4", embed=True)

In [ ]:
torch.save(model.state_dict(), "reinforce.pt")

In [ ]:
from model import compute_gae

rollout = sample_rollout(env, model, 24)
advantages, returns = compute_gae(rollout["rewards"], rollout["dones"], rollout["values"],
                                  rollout["next_value"], 0.99, 0.95)
advantages[:, 0]                                         # 第一只鸭子这 24 步的优势

In [ ]:
class A2C(Reinforce):
    lam = 0.95
    value_loss_coef = 1.0

    def prepare(self, rollout):
        # 超时只是截断，不是终点：把 critic 对那个状态的估值折扣后补进奖励
        rewards = rollout["rewards"] + self.gamma * rollout["values"].squeeze(-1) * rollout["time_outs"]
        advantages, returns = compute_gae(rewards, rollout["dones"], rollout["values"],
                                          rollout["next_value"], self.gamma, self.lam)
        return {**flat(rollout, "obs", "critic_obs", "actions"),
                "advantages": normalize(advantages).flatten(), "returns": returns.flatten(0, 1)}

    def loss(self, batch):
        values = self.model.value(batch["critic_obs"])
        return super().loss(batch) + self.value_loss_coef * self.value_loss(values, batch)

    def value_loss(self, values, batch):
        return (values - batch["returns"]).pow(2).mean()        # critic 的估值往价值目标靠

In [ ]:
fit(A2C(env, model), 1500)

In [ ]:
def load(path):
    policy = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")
    policy.load_state_dict(torch.load(path))
    return policy


def compare(policies):
    counter = env.unwrapped.common_step_counter          # 都对齐到这一档课程表，同一个评测
    for name, policy in policies.items():
        result = evaluate(policy.act_inference, SKILLS["walk"], hw, counter)
        print(name, result["falls"], round(result["mean_step_reward"], 3))

In [ ]:
compare({"REINFORCE": load("../S03-policy-gradient/reinforce.pt"), "A2C": model})

In [ ]:
record_video(model.act_inference, SKILLS["walk"], hw, env.unwrapped.common_step_counter, Path("a2c.mp4"))
Video("a2c.mp4", embed=True)

In [ ]:
torch.save(model.state_dict(), "a2c.pt")